In [ ]:
import pandas as pd
import re

def parse_trench_file(filename, lon_range=None, lat_range=None):
    """
    Parse a trench data file and optionally filter by geographic bounds.
    
    Args:
        filename (str): Path to the trench data file
        lon_range (tuple): (min_lon, max_lon) to filter longitude, optional
        lat_range (tuple): (min_lat, max_lat) to filter latitude, optional
    
    Returns:
        list: List of dictionaries, each containing segment info and coordinates
    """
    segments = []
    current_segment = None
    
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
            
            # Check if this is a segment header line (starts with >)
            if line.startswith('>'):
                # Save previous segment if it exists and has coordinates
                if current_segment and current_segment['coordinates']:
                    segments.append(current_segment)
                
                # Start new segment
                # Extract segment ID and name
                match = re.match(r'>\s*(\d+)\s+(.*)', line)
                if match:
                    segment_id = match.group(1)
                    segment_name = match.group(2).strip()
                else:
                    segment_id = "unknown"
                    segment_name = line[1:].strip()
                
                current_segment = {
                    'segment_id': segment_id,
                    'segment_name': segment_name,
                    'coordinates': []
                }
                continue
            
            # Try to parse coordinate line
            if current_segment is not None:
                try:
                    coords = line.split()
                    if len(coords) >= 2:
                        longitude = float(coords[0])
                        latitude = float(coords[1])
                        
                        # Apply geographic filtering if specified
                        if lon_range and not (lon_range[0] <= longitude <= lon_range[1]):
                            continue
                        if lat_range and not (lat_range[0] <= latitude <= lat_range[1]):
                            continue
                        
                        current_segment['coordinates'].append({
                            'lon': longitude,
                            'lat': latitude
                        })
                except (ValueError, IndexError):
                    # Skip lines that can't be parsed as coordinates
                    continue
    
    # Don't forget the last segment
    if current_segment and current_segment['coordinates']:
        segments.append(current_segment)
    
    return segments

def segments_to_dataframe(segments):
    """
    Convert segments list to a pandas DataFrame.
    
    Args:
        segments (list): List of segment dictionaries from parse functions
    
    Returns:
        pandas.DataFrame: DataFrame with columns for segment info and coordinates
    """
    data = []
    for segment in segments:
        for coord in segment['coordinates']:
            data.append({
                'segment_id': segment['segment_id'],
                'segment_name': segment['segment_name'],
                'lon': coord['lon'],
                'lat': coord['lat']
            })
    
    return pd.DataFrame(data)

def filter_segments_by_bounds(segments, lon_range=None, lat_range=None):
    """
    Filter existing segments by geographic bounds.
    
    Args:
        segments (list): List of segment dictionaries
        lon_range (tuple): (min_lon, max_lon) to filter longitude
        lat_range (tuple): (min_lat, max_lat) to filter latitude
    
    Returns:
        list: Filtered segments
    """
    filtered_segments = []
    
    for segment in segments:
        filtered_coords = []
        for coord in segment['coordinates']:
            lon = coord['lon']
            lat = coord['lat']
            
            # Check if coordinate is within bounds
            if lon_range and not (lon_range[0] <= lon <= lon_range[1]):
                continue
            if lat_range and not (lat_range[0] <= lat <= lat_range[1]):
                continue
            
            filtered_coords.append(coord)
        
        # Only include segment if it has coordinates within bounds
        if filtered_coords:
            filtered_segment = segment.copy()
            filtered_segment['coordinates'] = filtered_coords
            filtered_segments.append(filtered_segment)
    
    return filtered_segments

In [ ]:
import pygmt

# Load the grd file
grd_file1 = "cam_slab2_dep_02.24.18.grd"  # Replace with your actual file path
grd_file2 = "Kyriakopoulos_etal_JGR_2015_slabInterface.grd"  # Replace with your actual file path

# Get region info directly from the grid file
# region = pygmt.grdinfo(grid=grd_file, region=True)
# region=[273.5, 276, 9, 11.5]    # suitable region of interest
# region=[271.5, 276, 8, 11.5]    # suitable region for chopping the plate interface grid file 
region=[-90, -82, 6, 14]    # suitable region for chopping the plate interface grid file 

# load trench file
trench_file = "trench.gmt.inuse"

# Filter by both longitude and latitude
segments = parse_trench_file(trench_file, 
                           lon_range=(region[0], region[1]), 
                           lat_range=(region[2], region[3]))

# Convert to DataFrame for analysis
trench = segments_to_dataframe(segments)

# print(trench)

# Chose a survey line with start point A and end point B
lonA, latA, lonB, latB = -87+360, 10, -85.5+360, 11.5
lonC, latC, lonD, latD = -85.4+360, 8.8, -85.4+360+1.5, 8.8+1.5

# Create the figure
fig = pygmt.Figure()
pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p",
             FORMAT_GEO_MAP="ddd.xF")

fig.basemap(region=region, projection="M5i", frame=["a0.5f0.1", "WSne+tSlab2.0"])
fig.coast(borders=1, shorelines="0.25p,black", area_thresh=20, land="lightgray", water=None, resolution="h", 
          map_scale="g273.8/9.2+c9.2+w20k")   #land="251/250/240", water="241/246/247"

# Plot the contours from the grid
fig.grdcontour(
    grid=grd_file1,
    # region=region,
    # projection="M5i",     # Mercator projection with 6-inch width
    # levels=4,          # Contour interval in depth units (e.g., 10 km)
    # levels=10,          # Contour interval in depth units (e.g., 10 km)
    levels=5,          # Contour interval in depth units (e.g., 10 km)
    # levels=2.5,          # Contour interval in depth units (e.g., 10 km)
    limit="-120/0",
    # annotation="20+f10p+n0.5c/0c",  # Annotate every 10-unit contour line with 10-point font
    annotation="5+f8p,black",
    pen="0.75p,black",    # Line style for contours
)
fig.plot(x=trench['lon'], y=trench['lat'], pen="1.5p,black", style="f0.4i/0.1i+l+t", fill="black")
fig.plot(x=[lonA, lonB], y=[latA, latB], pen="1.5p,blue")
fig.plot(x=[lonC, lonD], y=[latC, latD], pen="1.5p,blue")

### panel 2, comparison with another local model 
# fig.shift_origin(yshift="h-9i")
fig.shift_origin(xshift="w+0.6i")

fig.basemap(region=region, projection="M5i", frame=["a0.5f0.1", "WSne+tKyriakopoulos et al., (2015)"])
fig.coast(borders=1, shorelines="0.25p,black", area_thresh=20, land="lightgray", water=None, resolution="h", 
          map_scale="g273.8/9.2+c9.2+w20k")   #land="251/250/240", water="241/246/247"
fig.grdcontour(grid=grd_file2, levels=4, limit="0/120", annotation="20+f10p,black", pen="0.75p,black")
fig.plot(x=trench['lon'], y=trench['lat'], pen="1.5p,black", style="f0.4i/0.1i+l+t", fill="black")

# Display the plot
fig.show()

In [ ]:
region1=[-86.75+360, -84.5+360, 8.75, 11.25] 

# Create the figure
fig = pygmt.Figure()
pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p",
             FORMAT_GEO_MAP="ddd.xF")

fig.basemap(region=region1, projection="M5i", frame=["a0.5f0.1", "WSne+tSlab2.0"])
fig.coast(borders=1, shorelines="0.25p,black", area_thresh=20, land="lightgray", water=None, resolution="h", 
          map_scale="g273.8/9.2+c9.2+w20k")   #land="251/250/240", water="241/246/247"

# Plot the contours from the grid
fig.grdcontour(grid=grd_file1, levels=5, limit="-60/0", annotation="10+f10p+n0.5c/0c", pen="0.75p,black")
fig.plot(x=trench['lon'], y=trench['lat'], pen="1.5p,black", style="f0.4i/0.1i+l+t", fill="black")
fig.plot(x=[lonA, lonB], y=[latA, latB], pen="1.5p,blue")
fig.plot(x=[lonC, lonD], y=[latC, latD], pen="1.5p,blue")

# ### panel 2, comparison with another local model 
# # fig.shift_origin(yshift="h-9i")
# fig.shift_origin(xshift="w+0.6i")

# fig.basemap(region=region1, projection="M5i", frame=["a0.5f0.1", "WSne+tKyriakopoulos et al., (2015)"])
# fig.coast(borders=1, shorelines="0.25p,black", area_thresh=20, land="lightgray", water=None, resolution="h", 
#           map_scale="g273.8/9.2+c9.2+w20k")   #land="251/250/240", water="241/246/247"
# fig.grdcontour(grid=grd_file2, levels=5, limit="0/80", annotation="5+f8p,black", pen="0.75p,black")
# fig.plot(x=trench['lon'], y=trench['lat'], pen="1.5p,black", style="f0.4i/0.1i+l+t", fill="black")

# Display the plot
fig.show()

fig.savefig("nicoya_plate_contours.pdf")

In [ ]:
# # Generate points along a great circle corresponding to the survey line and store them
# # in a pandas.DataFrame
# track_df = pygmt.project(
#     center=[lonA, latA],  # Start point of survey line (longitude, latitude)
#     endpoint=[lonB, latB],  # End point of survey line (longitude, latitude)
#     generate=0.1,  # Output data in steps of 0.1 degrees
# )

# # Extract the elevation at the generated points from the downloaded grid and add it as
# # new column "elevation" to the pandas.DataFrame
# track_df = pygmt.grdtrack(grid=grd_file1, points=track_df, newcolname="elevation")

# print(track_df)

In [ ]:
input_grd = "cam_slab2_dep_02.24.18.grd"
output_contour = "nicoya_slab2_100_10_10.txt"
output_contour = "nicoya_slab2_100_5_5.txt"
region="271.5/276/8/11.5"     # suitable region for chopping the plate interface grid file 

# !gmt grdcontour {input_grd} -C10 -L-100/10 -D{output_contour} -R{region} -V
!gmt grdcontour {input_grd} -C5 -L-100/0 -D{output_contour} -R{region} -V


In [ ]:
# Load the grd file
grd_file = "Kyriakopoulos_etal_JGR_2015_slabInterface.grd"  # Replace with your actual file path

# Get region info directly from the grid file
# region = pygmt.grdinfo(grid=grd_file, region=True)



In [ ]:
# trench_file = "PB2002_boundaries.dig.txt"
# # trench_file = "trench.gmt.txt"
# trench_file = "trench.gmt"

region="272/276/8/11.5"     # suitable region for chopping the plate interface grid file 
!gmt psxy {trench_file} -R{region} -JM5i -Sf0.1i/0.05i+l+t -Gblack -W2p,black -Baf > trench.ps
!ps2pdf trench.ps